### Old Elevation Function
#### With Bing

In [ ]:
import requests
import json
import time
def get_resource(r):
 
    resourceSets = json.loads(r.text)
 
    try:
        for resource in resourceSets['resourceSets']:
            return [rsc for rsc in resource['resources']]
    except KeyError:
        try:
            resourcesSets = resourceSets['Response']['ResourceSets']
            resourceSet = resourcesSets['ResourceSet']
            if isinstance(resourceSet, dict):
                return resourceSet['Resources']
            elif isinstance(resourceSet, list):
                return [rsc for rsc in resourceSet['Resources']]
        except KeyError:
            print(KeyError)
 
def get_elevations(point_list):
    
    ## BLOCKED KEYS 
    # key = 'Ap2jFsGK3XZxoJYoaoE8pMLEk2B4x8OqgkolcqfDPwN-24_V9Vd__m5YOCyXPXch'
    # key = 'ApDAnNZWDC_B1UmC_x_7btm-Nyql_Z6dcAmKnEpapSQDoM0-HjLLgBy6m-xIFNNR'
    # key = 'AkahIoYRb6rMD9hwACUWOBcv-B91nQ-LvoKSC9qdZzeQQc8QkZVarencGNi-Gpjr'

    ## NEW KEY##
    key='gZ3qi3ybxcZbEQrI3tSi~o4Gzhic8SgRxkyh9XRkQow~Ao8Gtd3pYUnZ3cobxWhTsz6wT57PbKbozd5jmJ4kZoL5LCuVNj9hasMQgm_7GC6v' ## NEW KEY [USE WITH CAUTION and keep time interval for multiple hits. Best to avoid multiple hit]
    str_point_query = ','.join([str(i) for i in point_list])
    api_query = 'http://dev.virtualearth.net/REST/v1/Elevation/List?points=' + str_point_query +'&heights=sealevel&key='+ key + '' #}&heights={}
    # api_query = 'http://dev.virtualearth.net/REST/v1/Elevation/List?points=' + str_point_query +'&heights=ellipsoid&key='+ key + '' #}&heights={}
    r = requests.get(api_query)
    # r.status_code
    if not r.status_code == 200:
        raise r.raise_for_status()
 
    resource = get_resource(r)
 
    elevation_list = resource[0]['elevations']
 
    return elevation_list,r.status_code
 
lat_long_list = [(-33.93631241196875,18.45023227330877),(-33.93891985523774,18.43850098810978),(-33.93177160076838,18.42758269803204),(-33.94261545048327,18.45850301812901),(-33.965526,18.377388),(3.182566,101.682517)]
for each in lat_long_list:
    elev = get_elevations([each[0],each[1]])
    print(elev)
    time.sleep(10)



                          
# [-33.93891985523774,18.43850098810978]#[-33.93177160076838,18.42758269803204]#[-33.94261545048327,18.45850301812901]#[-33.965526,18.377388]#[3.182566,101.682517]


### Open Elevation API
#### [API Doc Link](https://open-elevation.com/)

In [ ]:
# point_listOftuple - in (lat,long) format
import urllib.request
def get_elevations_open(point_listOftuple):
# point_listOftuple = [(-33.93631241196875,18.45023227330877)]
    #CONSTRUCT JSON
    d_ar=[{}]*len(point_listOftuple)
    for i in range(len(point_listOftuple)):
        d_ar[i]={"latitude":point_listOftuple[i][0],"longitude":point_listOftuple[i][1]}
    location={"locations":d_ar}
    json_data=json.dumps(location,skipkeys=int).encode('utf8')

    #SEND REQUEST 
    url="https://api.open-elevation.com/api/v1/lookup"
    response = urllib.request.Request(url,json_data,headers={'Content-Type': 'application/json'},method="POST")
    fp=urllib.request.urlopen(response)

    #RESPONSE PROCESSING
    res_byte=fp.read()
    res_str=res_byte.decode("utf8")
    js_str=json.loads(res_str)
    #print (js_mystr)
    fp.close()

    #GETTING ELEVATION 
    response_len=len(js_str['results'])
    elev_list=[]
    for j in range(response_len):
        elev_list.append(js_str['results'][j]['elevation'])


    return elev_list

In [ ]:
lat_long_list = [(-33.93631241196875,18.45023227330877),(-33.93891985523774,18.43850098810978),(-33.93177160076838,18.42758269803204),(-33.94261545048327,18.45850301812901),(-33.965526,18.377388),(3.182566,101.682517)]
for each in lat_long_list:
    elev = get_elevations_open([each])
    print(elev)
    time.sleep(10)
# elev = get_elevations_open(point_listOftuple)
# elev

### OpenMeteo

### [API LINK](https://open-meteo.com/en/docs/elevation-api)

In [ ]:
# point_listOftuple - in (lat,long) format


import requests
def get_elevations_openMeteo(point_listOftuple):
    list_of_lat = [point_listOftuple[i][0] for i in range(len(point_listOftuple)) ]
    list_of_lon = [point_listOftuple[i][1] for i in range(len(point_listOftuple)) ]

    url="https://api.open-meteo.com/v1/elevation"
    data = {'latitude': list_of_lat, 'longitude': list_of_lon}
    response = requests.post(url, data=data)
    # print(response.status_code)
    # print(response.content)
    res_str=response.content.decode("utf8")
    js_str=json.loads(res_str)
    if "elevation" in js_str.keys():
        elev_list = js_str["elevation"]
    else:
        elev_list = []


    return elev_list,response.status_code

In [257]:
lat_long_list = [(-33.93631241196875,18.45023227330877),(-33.93891985523774,18.43850098810978),(-33.93177160076838,18.42758269803204),(-33.94261545048327,18.45850301812901),(-33.965526,18.377388),(3.182566,101.682517)]

import pandas as pd
import random
import time

SUBGRID = "SUBGRID_DATA_IMAGEBASED_BLR"
df = pd.read_csv("C:/IMAGE/Foliage/image_api/subgrids/"+SUBGRID+".txt", sep='\t')

lat_long_list = []

for i in df.index:
    lat_long_list.append((df["PT_LAT"][i], df["PT_LON"][i]))

print(lat_long_list[:10])

[(13.12186479, 77.42433048), (13.12186479, 77.42477964), (13.12186479, 77.42522879), (13.12186479, 77.42567795), (13.12186479, 77.42612711), (13.12186479, 77.42657627), (13.12186479, 77.42702543), (13.12186479, 77.42747458), (13.12186479, 77.42792374), (13.12186479, 77.4283729)]


In [ ]:
list_elev = []
for each in lat_long_list[:5]:
    elev = get_elevations_openMeteo([each])
    print(elev)

    list_elev.append(elev[0][0])

    time.sleep(random.randrange(random.randrange(10, 20), random.randrange(30, 90)))

In [269]:
count = len(lat_long_list[:20])
print(count)
for i in range(count):
    counter = random.randrange(random.randrange(1, 2), random.randrange(2, 5))
    
    if i != count-1:
        print("HIT", i+1, "DELAY", counter)
        time.sleep(counter)
    else:
        print("HIT", i+1, "DELAY", 0)
print("DONE")

20
HIT 1 DELAY 1
HIT 2 DELAY 1
HIT 3 DELAY 1
HIT 4 DELAY 1
HIT 5 DELAY 1
HIT 6 DELAY 1
HIT 7 DELAY 2
HIT 8 DELAY 3
HIT 9 DELAY 1
HIT 10 DELAY 2
HIT 11 DELAY 1
HIT 12 DELAY 3
HIT 13 DELAY 2
HIT 14 DELAY 1
HIT 15 DELAY 1
HIT 16 DELAY 1
HIT 17 DELAY 1
HIT 18 DELAY 2
HIT 19 DELAY 2
HIT 20 DELAY 0
DONE


In [302]:
timer = []
for i in range(1000):
    t = random.randrange(random.randrange(1, 5), random.randrange(8, 15))
    #t = random.randrange(5, 15)
    timer.append(t)

sec = sum(timer)/len(timer)
print("Avg time per hit: ", sec)
print("Total time in days: ", sec*(538200+1367300+805800)/(60*60*24))

Avg time per hit:  6.409
Total time in days:  201.11946412037037


In [ ]:
print(list_elev)